# Run the CUDA GEMM benchmark on a free Colab T4

1. **Runtime → Change runtime type → T4 GPU**
2. Run all cells. Results are saved to `results/` — download them and commit to the repo.

In [ ]:
!nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv

In [ ]:
!git clone --depth 1 https://github.com/LakshikaMahipala/edge-ml-systems.git
%cd edge-ml-systems/projects/01-cuda-gemm-optimization

In [ ]:
!cmake -S . -B build -DCMAKE_CUDA_ARCHITECTURES=75 && cmake --build build -j

In [ ]:
!mkdir -p results
!./build/vector_add | tee results/vector_add_t4.csv
!./build/gemm_bench 256,512,1024,2048,4096 5 20 | tee results/gemm_t4.csv

In [ ]:
!python scripts/plot_results.py results/gemm_t4.csv results/gemm_t4.png
from IPython.display import Image
Image('results/gemm_t4.png')

In [ ]:
import pandas as pd
pd.read_csv('results/gemm_t4.csv', comment='#').pivot(index='M', columns='kernel', values='gflops').round(0)

In [ ]:
# Optional: profile one kernel with Nsight Compute (may be blocked on some Colab VMs)
!ncu --kernel-name regex:register_blocked --launch-count 1 --section SpeedOfLight ./build/gemm_bench 2048 0 1 || echo 'ncu not permitted here'

In [ ]:
from google.colab import files
for f in ['results/gemm_t4.csv','results/gemm_t4.png','results/vector_add_t4.csv']: files.download(f)